In [ ]:
import os, warnings, collections
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
data_root = "/kaggle/input/competitions/low-cost-motor-imagery-decoding-for-rehab-cross-subject"
test_dir = os.path.join(data_root, "Cross participant test set")
train_dir = os.path.join(data_root, "Training set")
test_files = sorted(os.listdir(test_dir))
print("test files:", test_files)

def marker_stats(path, label):
    df = pd.read_csv(path)
    m = df["Marker_val"].astype(str)
    m = m[m != "nan"].tolist()
    counter = collections.Counter(m)
    # gaps between consecutive markers (in samples)
    idx = df.index[df["Marker_val"].notna()].tolist()
    gaps = np.diff(idx)
    # cue->cue_stop gaps
    vals = [str(v) for v in df["Marker_val"].tolist()]
    cue_gaps = []
    for i, v in enumerate(vals):
        if v == "cue_start":
            j = vals[i+1:].index("cue_stop")
            cue_gaps.append(i + 1 + j - i)
    print(label, dict(counter))
    print("  marker gaps: mean %.1f median %.1f min %d max %d" % (gaps.mean(), np.median(gaps), gaps.min(), gaps.max()))
    if cue_gaps:
        print("  cue->cue_stop gaps (s): mean %.2f min %.2f max %.2f" % (np.mean(cue_gaps)/250, np.min(cue_gaps)/250, np.max(cue_gaps)/250))
    n = len(cue_gaps) if cue_gaps else 0
    print("  n cues:", n)

for f in test_files:
    marker_stats(os.path.join(test_dir, f), "TEST", )
# reference: one train file
marker_stats(os.path.join(train_dir, "S001-001_eeg.csv"), "TRAIN-ref")
print("MARKER DIAGNOSTICS DONE")